<a href="https://colab.research.google.com/github/2303a51355/High-performance-computing/blob/main/18_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%%bash
cat > race_condition.c << EOF
#include <stdio.h>
#include <pthread.h>
#define NUM_THREADS 10
#define INCREMENTS 100000
int counter = 0;
void* worker(void* arg) {
    for (int i = 0; i < INCREMENTS; i++) {
        counter++;  // race condition
    }
    return NULL;
}
int main() {
    pthread_t threads[NUM_THREADS];

    for (int i = 0; i < NUM_THREADS; i++) {
        pthread_create(&threads[i], NULL, worker, NULL);
    }

    for (int i = 0; i < NUM_THREADS; i++) {
        pthread_join(threads[i], NULL);
    }

    printf("Final Counter Value: %d\n", counter);
    return 0;
}
EOF

gcc race_condition.c -o race_condition -pthread
./race_condition

Final Counter Value: 783692


In [9]:
%%bash

cat > race_condition_mutex.c << EOF
#include <stdio.h>
#include <pthread.h>

#define NUM_THREADS 10
#define INCREMENTS 100000

int counter = 0;
pthread_mutex_t lock;

void* worker(void* arg) {
    for (int i = 0; i < INCREMENTS; i++) {
        pthread_mutex_lock(&lock);   // critical section start
        counter++;
        pthread_mutex_unlock(&lock); // critical section end
    }
    return NULL;
}

int main() {
    pthread_t threads[NUM_THREADS];

    pthread_mutex_init(&lock, NULL);

    for (int i = 0; i < NUM_THREADS; i++) {
        pthread_create(&threads[i], NULL, worker, NULL);
    }

    for (int i = 0; i < NUM_THREADS; i++) {
        pthread_join(threads[i], NULL);
    }

    pthread_mutex_destroy(&lock);

    printf("Final Counter Value: %d\n", counter);
    return 0;
}
EOF

gcc race_condition_mutex.c -o race_condition_mutex -pthread
./race_condition_mutex

Final Counter Value: 1000000
